In [15]:
!pip install langchain-text-splitters
!pip install reportlab
!pip install fpdf
!pip install pypdf
!pip install langchain-community
!pip install mysql-connector-python

In [19]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # Disable widgets

import mysql.connector
import pandas as pd
from fpdf import FPDF
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline


# ------------------------------------------------------------
# 1. CONNECT TO MYSQL
# ------------------------------------------------------------
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="S@i300498",
    database="shop_hunt"
)

df = pd.read_sql("SELECT * FROM product_inventory;", conn)

# rename safe column names
df.columns = [
    "record_id","shop_name","shop_owner","shop_address","product_name",
    "product_brand","product_mrp","product_size","quantity","selling_price",
    "manufacture_date","expiry_date","is_available","stock_status",
    "created_at","last_updated"
]

# Create product list for strict checking (important!)
product_list = df['product_name'].str.lower().tolist()


# ------------------------------------------------------------
# 2. CREATE PDF FROM DATABASE
# ------------------------------------------------------------
def create_pdf_from_df(df, output_path):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    pdf.cell(0, 10, "Product Inventory Report", ln=True, align='C')
    pdf.ln(5)

    for index, row in df.iterrows():
        text = (
            f"Record ID: {row['record_id']}\n"
            f"Shop Name: {row['shop_name']}\n"
            f"Owner: {row['shop_owner']}\n"
            f"Address: {row['shop_address']}\n"
            f"Product Name: {row['product_name']}\n"
            f"Brand: {row['product_brand']}\n"
            f"MRP: {row['product_mrp']}\n"
            f"Size: {row['product_size']}\n"
            f"Quantity: {row['quantity']}\n"
            f"Selling Price: {row['selling_price']}\n"
            f"Manufacture Date: {row['manufacture_date']}\n"
            f"Expiry Date: {row['expiry_date']}\n"
            f"Available: {row['is_available']}\n"
            f"Stock Status: {row['stock_status']}\n"
            f"Created: {row['created_at']}\n"
            f"Updated: {row['last_updated']}\n"
            "---------------------------------------------"
        )
        pdf.multi_cell(0, 8, text)
        pdf.ln(2)

    pdf.output(output_path, dest='F')


pdf_path = os.path.join(os.getcwd(), "product_report.pdf")
create_pdf_from_df(df, pdf_path)


# ------------------------------------------------------------
# 3. LOAD PDF USING LANGCHAIN
# ------------------------------------------------------------
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Remove line breaks for cleaner embedding
for doc in documents:
    doc.page_content = doc.page_content.replace("\n", " ").strip()


# ------------------------------------------------------------
# 4. SPLIT PDF INTO CHUNKS
# ------------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40
)
chunks = text_splitter.split_documents(documents)


# ------------------------------------------------------------
# 5. CREATE EMBEDDINGS + FAISS DB
# ------------------------------------------------------------
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)


# ------------------------------------------------------------
# 6. QA MODEL (FLAN-T5)
# ------------------------------------------------------------
qa_model = pipeline("text2text-generation", model="google/flan-t5-base")


# ------------------------------------------------------------
# 7. UPDATED QUESTION ANSWERING FUNCTION
# ------------------------------------------------------------
def answer_question(vector_db, question):
    question_lower = question.lower()

    # ---------- STRICT PRODUCT VALIDATION ----------
    matched_products = [p for p in product_list if p in question_lower]

    # Product does NOT exist
    if not matched_products:
        return "Product unavailable"

    # Product exists → get first match
    product_name = matched_products[0]

    # Lookup direct SQL row
    product_row = df[df['product_name'].str.lower() == product_name]

    if product_row.empty:
        return "Product unavailable"

    shop_name = product_row.iloc[0]['shop_name']
    shop_address = product_row.iloc[0]['shop_address']
    stock_status = product_row.iloc[0]['stock_status']

    # Return the shop info directly
    return f"""

Shop Name   : {shop_name}
Address     : {shop_address}
Stock Status: {stock_status}
"""


# ------------------------------------------------------------
# 8. USER QUERY
# ------------------------------------------------------------
question = input("Ask your question: ")
answer = answer_question(vector_db, question)
print("\nAssistant:", answer)

C:\Users\admine\AppData\Local\Temp\ipykernel_10440\3747500575.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM product_inventory;", conn)
Device set to use cpu


Ask your question:  samsung galaxy s24



Assistant: 

Shop Name   : shivam
Address     : bv nagar nellore
Stock Status: IN_STOCK

